# Lesson 01 - Introduktion till AI-agenter

Välkommen till den första lektionen i kursen **AI Agents for Beginners**!

En **AI-agent** är ett program som använder en stor språkmodell (LLM) som sin resonemangsmotor och kan utföra *åtgärder* i den verkliga världen — ringa API:er, fråga databaser eller köra kod — för att uppnå ett mål på uppdrag av en användare.

I denna anteckningsbok kommer du att skapa din första agent: en **Reseagent** som rekommenderar semesterresmål. Under tiden kommer du att lära dig hur du:

1. Ansluter till Azure AI Foundry Agent Service med hjälp av **Microsoft Agent Framework**.
2. Ger agenten ett **verktyg** — en vanlig Python-funktion som den kan anropa.
3. Kör agenten och inspekterar dess svar.
4. Strömmar agentens svar token för token.


## Setup

Innan du kör den här anteckningsboken, se till att du har:

1. **Ett Azure AI Foundry-projekt** med en distribuerad chattmodell (t.ex. `gpt-4o-mini`).
2. **Loggat in med Azure CLI** — kör `az login` i din terminal.
3. **Ställt in de nödvändiga miljövariablerna:**
   - `AZURE_AI_PROJECT_ENDPOINT` — din Azure AI Foundry projektendpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — namnet på din distribuerade modell.

Cellen nedan installerar de Python-paket du behöver.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Skapa din första agent

En agent behöver två saker:

- **Instruktioner** som berättar *vem den är* och *hur den ska bete sig* (en systemprompt).
- **Verktyg** — Python-funktioner dekorerade med `@tool` som agenten kan anropa för att hämta information eller utföra åtgärder.

Nedan definierar vi ett enkelt verktyg som returnerar en lista över populära semesterdestinationer. Agenten kommer att använda detta verktyg när en användare frågar efter resetips.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming-svar

För en mer interaktiv upplevelse kan du **strömma** agentens svar. Istället för att vänta på hela svaret levererar agenten textdelar allteftersom de genereras. Detta är särskilt användbart i chattgränssnitt där du vill visa resultatet i realtid.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

I den här lektionen lärde du dig att:

- **Skapa en leverantör** som ansluter till Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definiera ett verktyg** med hjälp av dekoratorn `@tool` så att agenten kan anropa dina Python-funktioner.
- **Köra agenten** med ett användarmeddelande och skriva ut dess svar.
- **Streama svar** för realtidsutdata.

I nästa lektion kommer vi att utforska agentbaserade ramverk mer i detalj och lära oss hur man ger agenter kraftfullare verktyg och möjlighet till flerstegsresonemang.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, bör du vara medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål ska anses vara den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
